# Data Analysis · Week 15, session 3 of 3
## Grouping, summarising and joining

**TIA502 · School of Business · Instructor David Escobar-Castillejos**

The eighty lines you wrote by hand in week 14 fit into eight today, and the numbers match to
the cent. You will see the PivotTable and the `VLOOKUP` written out, plus a check no
spreadsheet lets you run.

By the end of this notebook you will be able to:

1. Group and summarise in one line, with `groupby` followed by the column and the function.
2. Ask for several summaries at once with `agg`, using column names you choose.
3. Build a grid with `pivot_table`, including the row and column totals.
4. Join two tables with `merge`, and explain why the left mode is the safe one.
5. Audit a join with `indicator`, checking both directions before trusting it.

### How to use this notebook

Run the cells in order. This notebook does not depend on you having run session 15.2: the
second cell repeats the cleaning so you can open this one on its own.

Three cells fail on purpose and carry a comment saying so.

---
## Setup

In [ ]:
import pandas as pd

print("pandas", pd.__version__)

In [ ]:
# Plumbing, not the lesson. This cell puts the three course CSVs within
# reach of pandas, and it never needs running again.
#
# It looks for them in the repository first, which is public and reads
# over a URL. If that does not answer, it rebuilds them right here from
# the course's fixed seed, so both routes produce identical files.
# Nothing ever has to be uploaded by hand.
import urllib.request
from pathlib import Path

BASE = ("https://raw.githubusercontent.com/Davidowa/learning-hub/main/"
        "docs/en/courses/python-course/06%20-%20Advanced/data/")
FILES = ["sales.csv", "regions.csv", "employees.csv"]


def _descargar():
    for nombre in FILES:
        with urllib.request.urlopen(BASE + nombre, timeout=15) as r:
            Path(nombre).write_bytes(r.read())


def _reconstruir_datos():
    """Write the three CSV files again from the course's fixed seed.

    They come out byte for byte identical to the ones in the repository,
    so the numbers on the slide still match the ones in the notebook.
    """
    import csv, random
    from datetime import date, timedelta

    rng = random.Random(20260808)
    REGIONS = ["North", "South", "Centre", "West"]
    CHANNELS = ["Retail", "Online", "Wholesale"]
    PRODUCTS = {"Espresso machine": 8990.0, "Coffee grinder": 2450.0,
                "Filter kettle": 1290.0, "Bean subscription": 690.0,
                "Travel mug": 349.0}
    RW = {"North": 1.30, "South": 0.80, "Centre": 1.55, "West": 0.95}
    CW = {"Retail": 1.00, "Online": 1.25, "Wholesale": 2.10}
    MW = [0.72, 0.78, 0.90, 0.95, 1.00, 1.05, 0.98, 0.92, 1.08, 1.15, 1.45, 1.60]

    rows, start = [], date(2025, 1, 6)
    for week in range(52):
        day = start + timedelta(weeks=week)
        for region in REGIONS:
            for _ in range(rng.randint(1, 2)):
                product = rng.choice(list(PRODUCTS))
                channel = rng.choice(CHANNELS)
                base = 9 * RW[region] * CW[channel] * MW[day.month - 1]
                units = max(1, round(rng.gauss(base, base * 0.28)))
                price = PRODUCTS[product] * rng.choice([1.0, 1.0, 1.0, 0.9, 0.85])
                rows.append({"date": day.isoformat(), "region": region,
                             "channel": channel, "product": product,
                             "units": str(units),
                             "unit_price": f"$ {price:,.2f}"})

    # the deliberate dirt: one region typed four ways, blank cells, and
    # rows captured twice
    for i in rng.sample(range(len(rows)), 24):
        rows[i]["region"] = rng.choice(["north", "NORTH", " North", "North "])
    for i in rng.sample(range(len(rows)), 11):
        rows[i]["units"] = ""
    for i in rng.sample(range(len(rows)), 7):
        rows.append(dict(rows[i]))
    rng.shuffle(rows)

    with open("sales.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["date", "region", "channel",
                                          "product", "units", "unit_price"])
        w.writeheader()
        w.writerows(rows)

    with open("regions.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["region", "manager", "country", "monthly_target"])
        w.writerows([["North", "Ana Robles", "Mexico", 480000],
                     ["South", "Luis Ferrer", "Mexico", 300000],
                     ["Centre", "Paula Ines", "Mexico", 560000],
                     ["West", "Marco Duarte", "Mexico", 360000],
                     ["East", "Sofia Lara", "Mexico", 220000]])

    AREAS = {
        "Sales": (["Account executive", "Sales analyst", "Sales manager"], 24000, 62000),
        "Marketing": (["Content specialist", "Campaign analyst", "Brand manager"], 22000, 58000),
        "Finance": (["Accounts clerk", "Financial analyst", "Controller"], 26000, 74000),
        "People": (["Recruiter", "People analyst", "People manager"], 21000, 55000),
        "Operations": (["Warehouse lead", "Logistics analyst", "Operations manager"], 20000, 60000),
    }
    CITIES = ["Mexico City", "Guadalajara", "Monterrey", "Queretaro"]
    emp = []
    for n in range(1, 121):
        area = rng.choice(list(AREAS))
        titles, low, high = AREAS[area]
        idx = rng.choices([0, 1, 2], weights=[5, 3, 1])[0]
        tenure = rng.randint(2, 132)
        salary = round(low + (high - low) * (idx / 2) * rng.uniform(0.82, 1.10)
                       + tenure * 45, -2)
        emp.append({"employee_id": f"E{n:04d}", "area": area,
                    "job_title": titles[idx], "city": rng.choice(CITIES),
                    "tenure_months": tenure, "monthly_salary": int(salary)})
    with open("employees.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(emp[0]))
        w.writeheader()
        w.writerows(emp)


try:
    _descargar()
    print("Data read from the repository.")
except Exception:
    _reconstruir_datos()
    print("The repository did not answer. Data rebuilt in this session.")

print("Ready:", ", ".join(FILES))

### The cleaning, again and without explanation

Grouping over dirty data is the first error of this session, so the file gets cleaned before
anything touches it. If something in this cell is unclear, session 15.2 explains it step by
step.

In [ ]:
# The cleaning from session 15.2, in one cell. Nothing new here: it is so this
# notebook opens on its own, without depending on you having run the last one.
sales = pd.read_csv("sales.csv").drop_duplicates()
sales["region"] = sales["region"].str.strip().str.title()
sales["unit_price"] = (sales["unit_price"]
                       .str.replace("$", "", regex=False)
                       .str.replace(",", "", regex=False)
                       .str.strip()
                       .astype(float))
sales["date"] = pd.to_datetime(sales["date"])
sales = sales.dropna(subset=["units"])
sales["units"] = sales["units"].astype(int)
sales["amount"] = sales["units"] * sales["unit_price"]

sales.to_csv("sales_clean.csv", index=False)
print(f"{len(sales)} clean rows, total {sales['amount'].sum():,.2f}")

---
# Block 1 · Grouping

`groupby` does exactly what dragging a field into a PivotTable does: it splits the rows into
buckets that share a value, applies a summary to each bucket, and puts the results back
together as a table.

Split, summarise, put back together. That is all of it.

## Eighty lines, or eight

First the week 14 version, with a dictionary and a loop. It really runs, over the clean file
you just wrote.

In [ ]:
import csv
from collections import defaultdict

by_region_manual = defaultdict(float)

with open("sales_clean.csv", encoding="utf-8") as f:
    for record in csv.DictReader(f):
        by_region_manual[record["region"]] += float(record["amount"])

for region in sorted(by_region_manual, key=by_region_manual.get, reverse=True):
    print(f"{region:8} {by_region_manual[region]:>12,.2f}")

Now the same thing with pandas.

In [ ]:
by_region = sales.groupby("region")["amount"].sum()

print(by_region.sort_values(ascending=False).round(2))

The same four totals, to the cent. The difference is that one reads at a glance and the other
has to be checked line by line before you believe it.

Worth confirming rather than taking my word for it.

In [ ]:
same = all(
    round(by_region_manual[region], 2) == round(by_region[region], 2)
    for region in by_region.index
)
print("Do the four totals match?", same)

The result of `groupby` is a `Series` whose index is the thing you grouped by, so everything
from session 15.1 still applies here.

In [ ]:
print("Best region:", by_region.idxmax())
print("Its share of the year:", f"{by_region.max() / sales['amount'].sum():.1%}")
print()
print("All four, in thousands:")
print((by_region.sort_values(ascending=False) / 1000).round(0))

## Several summaries at once

`agg` takes a list of functions and gives back a column for each. It answers how much, how
many times and how big in a single pass.

In [ ]:
summary = sales.groupby("region")["amount"].agg(["sum", "count", "mean"]).round(2)

print(summary.sort_values("sum", ascending=False))

Different summaries for different columns, with names you choose. The pattern is
`new_name=("source column", "function")`, and it is how a report table gets built in a single
statement.

In [ ]:
detailed = sales.groupby("region").agg(
    revenue=("amount", "sum"),
    units_sold=("units", "sum"),
    sales_made=("amount", "count"),
    average_sale=("amount", "mean"),
).round(2)

print(detailed.sort_values("revenue", ascending=False))

### Here is the story

Look at that table carefully before moving on. **North sells more than anyone, and Centre has
the highest average sale.** North made 92 sales averaging 47 thousand; Centre made 70
averaging 56 thousand.

They are two different businesses with the same apparent revenue, and that difference does
not show up in a total. `sum` answers magnitude and `count` answers frequency, which are the
two questions from week 9, and you need both to understand what happened.

In [ ]:
print("Sales made per region:")
print(detailed["sales_made"].sort_values(ascending=False))
print()
print("Average size of a sale:")
print(detailed["average_sale"].sort_values(ascending=False).round(0))

## Two grouping fields

Pass a list and the buckets become every combination of the two fields.

In [ ]:
by_region_channel = sales.groupby(["region", "channel"])["amount"].sum().round(2)

print(by_region_channel)

Twelve rows, one per combination. It reads as a long list, and that is exactly the problem the
next block solves.

---
# Block 2 · The grid

`pivot_table` lays those same numbers out as a grid, which is the shape you see on screen
when you open a PivotTable.

Four arguments, and each one matches something you would drag with the mouse:

| Argument | What it is | In the PivotTable |
|---|---|---|
| `index` | What goes down the side | The field you drag into rows |
| `columns` | What goes across the top | The field you drag into columns |
| `values` | What fills the cells | The values field |
| `aggfunc` | How they are summarised | Value field settings |

In [ ]:
grid = sales.pivot_table(
    index="region",      # what goes down the side
    columns="channel",   # what goes across the top
    values="amount",     # what fills the cells
    aggfunc="sum",       # how they are summarised
)

print((grid / 1000).round(0))

The same twelve numbers from the previous block, now readable at a glance. In thousands, so
they fit.

## The `aggfunc` trap

**Predict before you run.** What does `pivot_table` give back if you do not say `aggfunc`?

- **A.** The sum by region and channel.
- **B.** The mean, which is what it does by default.
- **C.** The row count.
- **D.** An error, because `aggfunc` is required.

In [ ]:
# FAILS ON PURPOSE. It raises nothing: it gives a different number, which is worse.
no_aggfunc = sales.pivot_table(
    index="region",
    columns="channel",
    values="amount",
)

print("Without aggfunc, Centre/Online cell:", round(no_aggfunc.loc["Centre", "Online"], 2))
print("With sum,        Centre/Online cell:", round(grid.loc["Centre", "Online"], 2))
print()
print("How many times bigger is the sum?",
      round(grid.loc["Centre", "Online"] / no_aggfunc.loc["Centre", "Online"], 1))

The answer is **B**. By default `pivot_table` averages, it does not add.

And there is the danger: it raises nothing, it gives back a grid with the same shape and the
same headers, holding numbers thirty times smaller. If you expected totals and did not say
`aggfunc`, your report is wrong and looks perfectly fine.

The factor they differ by is not a coincidence: it is how many sales fell into that cell.

## The totals

In [ ]:
with_totals = sales.pivot_table(
    index="region", columns="channel", values="amount",
    aggfunc="sum", margins=True, margins_name="Total",
)

print((with_totals / 1000).round(0))

`margins=True` adds the row and column totals, the way a PivotTable's grand total does. The
number in the bottom right corner has to match the table's total, and checking it is the
fastest way to know whether something got lost along the way.

In [ ]:
corner = with_totals.loc["Total", "Total"]
table = sales["amount"].sum()

print(f"Corner of the grid: {corner:,.2f}")
print(f"Total of the table: {table:,.2f}")
print("Match?", round(corner, 2) == round(table, 2))

## Grouping over time

A date column can be grouped by any part of itself. `.dt` reaches into the date the same way
`.str` reaches into text.

In [ ]:
sales["month"] = sales["date"].dt.month
monthly = sales.groupby("month")["amount"].sum().round(2)

print((monthly / 1000).round(0))
print()
print("Best month:", monthly.idxmax(), "| worst month:", monthly.idxmin())

December runs away with it, at more than double almost any other month, and July is the
weakest. That pattern only appears once you group: row by row in the table it is invisible.

Now, careful with the easy reading. The file carries a seasonal curve that rises in November
and December, and November still came out low. The reason is that a handful of espresso
machine sales, the expensive product, outweigh the seasonality of everything else. A total
hides its composition, and that is exactly why `agg` with `count` next to `sum` earns its
place.

In [ ]:
print("Revenue and number of sales per month:")
print(sales.groupby("month").agg(
    revenue=("amount", "sum"),
    sales_made=("amount", "count"),
    ticket=("amount", "mean"),
).round(0).sort_values("revenue", ascending=False).head())

December had an average ticket well above the rest, not many more sales. The month was not
better because things sold more often, it was better because they sold dearer.

Quarter, year and day of week work the same way.

In [ ]:
by_quarter = sales.groupby(sales["date"].dt.quarter)["amount"].sum().round(2)
print("By quarter, in thousands:")
print((by_quarter / 1000).round(0))

print()
print("By day of week, in thousands:")
print((sales.groupby(sales["date"].dt.day_name())["amount"].sum() / 1000).round(0))

Day of week comes back with a single value because the file was generated with one sale per
week, every Monday. It is a good reminder that a grouping does not invent variety where there
is none, and that it pays to look at the result before drawing a conclusion.

## The top of each group

A question that turns up in every report: which product sells most in each region. Group by
both, total, then take the largest of each region.

In [ ]:
product_region = sales.groupby(["region", "product"])["amount"].sum()
best_per_region = product_region.loc[product_region.groupby("region").idxmax()]

print(best_per_region.round(2))

The middle line reads from the inside out: `groupby("region").idxmax()` gives back, for each
region, the full label of its largest row, and `loc` goes and fetches those rows. It is the
same `idxmax` from session 15.1, applied to a two-level index.

---
# Block 3 · Joining two tables

`merge` is `VLOOKUP`, with two differences that matter. It brings every column across at once
instead of one per formula, and it tells you what did not match instead of leaving `#N/A`
scattered through the sheet.

| Mode | What it keeps | When |
|---|---|---|
| `left` | Every row on the left | The safe one, and the `VLOOKUP` lookalike |
| `inner` | Only the ones that match | When what does not cross does not matter |
| `right` | Every row on the right | Rare, it is a `left` backwards |
| `outer` | Everything from both sides | For auditing what did not match |

In [ ]:
regions = pd.read_csv("regions.csv")

print(regions)
print()
print("Sales:", sales.shape, "| Regions:", regions.shape)

`on` names the column both tables share. Every matching row of `regions` is attached to the
sales row, bringing all of its columns with it.

In [ ]:
joined = sales.merge(regions, on="region", how="left")

print("After the join:", joined.shape)
print(joined[["date", "region", "amount", "manager", "monthly_target"]].head(3))

`how="left"` keeps every sales row, whether or not the lookup found a match. That is the
`VLOOKUP` behaviour and it is the safe default: you never silently lose a sale because its
region was missing from the catalogue.

Watch the shape: 306 rows before, 306 after. Had that number changed, something happened that
needs understanding before you go on.

## The audit, which is what no spreadsheet lets you do

In [ ]:
audit = sales.merge(regions, on="region", how="outer", indicator=True)

print(audit["_merge"].value_counts())

`how="outer"` keeps everything from both sides, and the `_merge` column says where each row
came from. The two directions get checked separately and they mean different things.

**`right_only` at one** means the catalogue holds a region with no sales at all. That is
usually fine: a new location, or one that closed.

**`left_only` at zero** means no sale was left orphaned. That one matters: a sale whose region
the catalogue does not know is a data problem to report, not to paper over.

In [ ]:
orphans = audit[audit["_merge"] == "right_only"]["region"].unique()
print("Catalogue regions with no sales:", list(orphans))

uncatalogued = audit[audit["_merge"] == "left_only"]["region"].unique()
print("Sales whose region the catalogue does not know:", list(uncatalogued) or "none")

With a formula you would have to count the `#N/A` by hand, and only in one direction. Here the
count comes included and it covers both.

## Using what the join brought

Now that every row knows its target, the comparison is an ordinary column.

In [ ]:
monthly_region = (
    joined.assign(month=joined["date"].dt.month)
    .groupby(["region", "manager", "monthly_target", "month"])["amount"]
    .sum()
    .reset_index()
)

monthly_region["hit_target"] = monthly_region["amount"] >= monthly_region["monthly_target"]
monthly_region["attainment"] = (monthly_region["amount"] / monthly_region["monthly_target"]).round(3)

print(monthly_region.head())

`reset_index` turns the grouping index back into ordinary columns. Without it, `region`,
`manager`, `monthly_target` and `month` would still be index and could not be used in a
comparison.

And with that the scoreboard can be built.

In [ ]:
scoreboard = (
    monthly_region.groupby(["region", "manager"])
    .agg(months=("hit_target", "count"),
         months_on_target=("hit_target", "sum"),
         mean_attainment=("attainment", "mean"))
    .round(3)
    .sort_values("mean_attainment", ascending=False)
)

print(scoreboard)

No region hit its target more than three months out of twelve, and the best one averages 76 %
attainment. Neither table says that on its own: `sales` does not know the targets and
`regions` does not know the sales. It comes from having joined them.

## When the key is named differently in each table

Name both sides. Here both are called `region`, so the example renames one on the way in to
show the shape.

In [ ]:
codes = regions.rename(columns={"region": "region_code"})
example = sales.merge(codes, left_on="region", right_on="region_code", how="left")

print("Joined on differently named keys:", example.shape)
print(example[["region", "region_code", "manager"]].head(3))

Note that both key columns survived, `region` and `region_code`, holding the same content.
That is normal, and if they get in the way they come off with `drop`.

## Exporting

The analysis ends where it started, as a file somebody else can open.

In [ ]:
scoreboard.to_csv("scoreboard.csv")

print("Wrote scoreboard.csv")
print(open("scoreboard.csv", encoding="utf-8").read())

For Excel, which is where this usually has to end up, an `ExcelWriter` handles several sheets
at once. It needs the `openpyxl` package, which Colab already has installed.

In [ ]:
# If openpyxl were missing, pandas raises ImportError naming it. Colab ships with it.
try:
    with pd.ExcelWriter("report.xlsx") as writer:
        scoreboard.to_excel(writer, sheet_name="Scoreboard")
        monthly_region.to_excel(writer, sheet_name="Monthly", index=False)
        regions.to_excel(writer, sheet_name="Regions", index=False)
    print("Wrote report.xlsx with three sheets")
except ImportError as e:
    print("The .xlsx was not written:", e)

The files sit in the Colab session. To pull them down to your machine, the file panel on the
left has a download option in each file's menu.

---
## Four errors when grouping and joining

**Grouping without having cleaned.** Eight regions where there are four. The totals split and
each half looks perfectly reasonable. That is why the cleaning comes first in this notebook.

**Assuming `pivot_table` sums.** By default it averages. You have seen the number it gives,
and you have seen that it does not warn you.

**Joining with `inner` without noticing.** The rows that do not cross disappear silently, and
your total drops with nothing to explain it.

**Trusting a join without auditing it.** `indicator=True` costs one word and tells you exactly
how many rows were left loose, in both directions.

---
# Exercises

The solutions sit at the very bottom.

## Grouping

### Exercise 1 · Three simple groupings

On `sales`, work out and print:

1. Total revenue per channel, sorted highest to lowest.
2. How many units were sold of each product.
3. The average unit price per product, rounded to two decimals.

### Exercise 2 · The one-statement report

Use `agg` to build a per-product table carrying, with these exact names: `revenue`, `units`,
`sales`, `average_ticket` and `average_price`. Sort it by revenue.

Then answer in a comment which product to push if what you want is more revenue, and which
one if what you want is more sales.

### Exercise 3 · The grid that crosses time

Build a grid with the month down the rows, the region across the columns and revenue in the
cells, with totals. Divide it by a thousand and round it so it can be read.

Check that the corner matches the table's total.

## Joining

### Exercise 4 · The audited join, backwards

Join `regions` against `sales`, meaning with `regions` on the left, in `left` mode. Compare
how many rows come back against the join we did in class and explain in a comment why the
number is different.

### Exercise 5 · The invented region

Add a row by hand to a copy of `sales` with the region `"East Coast"`, which is not in the
catalogue. Run the audit and confirm that `left_only` is no longer zero.

Then say, in a comment, what you would do with that row if it turned up in your project.

### Exercise 6 · The month each region delivered

With `monthly_region`, find each region's best and worst month by attainment. Print region,
month and attainment for both.

Hint: `idxmax` inside a `groupby`, like the best-selling-product exercise.

## With your own data

### Exercise 7 · Answer your business question

With your clean file, answer the question you posed in week 1 using a grouping. Produce a
grid crossing two categories as well, and join it with a catalogue table if your case calls
for one.

If there is a join, it has to come audited with `indicator` and commented.

The test: sum the whole grid and compare it against the table's total. If they do not match,
something got lost.

---
## Three ideas to take away

**`groupby` is the PivotTable.** Split, summarise, put back together. The eighty lines of week
14, in eight, with the same totals to the cent.

**`pivot_table` averages by default.** If you wanted totals you have to say so, and forgetting
produces a number thirty times smaller that looks perfectly reasonable.

**A join gets audited.** `indicator` costs one word and answers, in both directions, which
rows found no partner.

Next session is charts. What a number has to look like for somebody to understand it without
you explaining it.

---
# Solutions

### Exercise 1

```python
print("Revenue per channel:")
print((sales.groupby("channel")["amount"].sum().sort_values(ascending=False) / 1000).round(0))

print("\nUnits per product:")
print(sales.groupby("product")["units"].sum().sort_values(ascending=False))

print("\nAverage unit price per product:")
print(sales.groupby("product")["unit_price"].mean().round(2).sort_values(ascending=False))
```

Wholesale takes more than half the revenue, and not because it sells more often but because
each sale is bigger. The same lesson again: magnitude and frequency are two questions.

### Exercise 2

```python
by_product = sales.groupby("product").agg(
    revenue=("amount", "sum"),
    units=("units", "sum"),
    sales=("amount", "count"),
    average_ticket=("amount", "mean"),
    average_price=("unit_price", "mean"),
).round(2)

print(by_product.sort_values("revenue", ascending=False))

# To lift revenue, push the Espresso machine: it has the highest ticket by a wide
# margin, so every extra sale counts for a lot. To lift the number of sales, push
# the Travel mug or the Bean subscription, the cheap ones, which is why they move
# most. Two different strategies, and the table separates them.
```

This table is the whole report in one statement. That you choose the column names is what
makes it a deliverable rather than an intermediate step.

### Exercise 3

```python
by_month = sales.pivot_table(
    index=sales["date"].dt.month,
    columns="region",
    values="amount",
    aggfunc="sum",
    margins=True,
    margins_name="Total",
)

print((by_month / 1000).round(0))

corner = by_month.loc["Total", "Total"]
print("\nDoes the corner match?", round(corner, 2) == round(sales["amount"].sum(), 2))
```

`index` accepts a Series computed on the spot, not only a column name. That is what saves you
from creating a `month` column that gets in the way afterwards.

### Exercise 4

```python
backwards = regions.merge(sales, on="region", how="left")

print("Sales merge regions:", len(sales.merge(regions, on="region", how="left")))
print("Regions merge sales:", len(backwards))
print(backwards[backwards["amount"].isna()][["region", "manager"]])

# One extra row comes back, 307 against 306. With regions on the left, how="left"
# keeps all five catalogue regions, East included, which has no sales at all. That
# row shows up with every sales column as NaN. It is not an error: it is the
# catalogue saying East exists and sold nothing.
```

Which table goes on the left is a decision, not a detail. The one on the left is the one
guaranteed to survive intact.

### Exercise 5

```python
invented = sales.copy()
row = sales.iloc[0].copy()
row["region"] = "East Coast"
invented = pd.concat([invented, row.to_frame().T], ignore_index=True)

check = invented.merge(regions, on="region", how="outer", indicator=True)
print(check["_merge"].value_counts())
print("\nSales with no region in the catalogue:")
print(check[check["_merge"] == "left_only"][["region", "amount"]])

# That row does not get deleted and does not get a region invented for it. It goes
# back to whoever captured the data, because "East Coast" could be East misspelled,
# a new location nobody registered, or a sale from another company. Those three get
# fixed differently and none of them gets fixed by guessing.
```

This exercise is what justifies always auditing. One row in 307 does not move the total enough
for anyone to notice, and it is still wrong data.

### Exercise 6

```python
best = monthly_region.loc[monthly_region.groupby("region")["attainment"].idxmax()]
worst = monthly_region.loc[monthly_region.groupby("region")["attainment"].idxmin()]

print("Best month per region:")
print(best[["region", "manager", "month", "attainment"]].to_string(index=False))
print("\nWorst month per region:")
print(worst[["region", "manager", "month", "attainment"]].to_string(index=False))
```

`idxmax` over a `groupby` gives back one label per group, and `loc` turns them into whole
rows. It is the same pattern as the best-selling product, and once you recognise it you will
use it in nearly every report.

### Exercise 7

There is no published solution, because the file is different for everyone. It is graded on
three things: that the grouping answers the question you posed rather than another one, that
the audit is commented if there was a join, and that the grid's sum matches the table's total.